<a href="https://colab.research.google.com/github/tettehdaniel/python-labs/blob/main/Notebooks/Notebook18_Regularization_SV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 18 — Regularization Models

**Main Objective**

1. Lasso Regression
2. Ridge Regression
3. Elastic-Net (E-Net) Regression

## Setup

Please run the setup code below

In [1]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

from funs import *

In [2]:
import numpy as np
import polars as pl

from plotnine import *
from polars import col as c
theme_set(theme_minimal())

path = "https://raw.githubusercontent.com/ajcollins16/DSST289_Fall2026_AJC/refs/heads/main/"

In [3]:
dat = pl.read_csv(path + "Data/bodyfat.csv")

## Regularization Models

Today we are going to be using a body fat dataset, which contains measruements from 252 males. Specifically, this dataset contains a collection of physical measurements that can be used to predict an individual's percentage of body fat. Many of these measurements, such as weight and the circumferences of different parts of the body, provide overlapping information and may therefore be strongly related to one another. This makes the dataset particularly useful for exploring regularized regression models.

In the previous notebook, we evaluated predictive models by comparing their performance on training and testing data. In this notebook, we will extend linear regression by introducing regularization. Rather than allowing the model to freely choose coefficients that minimize error on the training data, regularized models apply a penalty that shrinks the coefficients. We will compare three approaches: lasso regression, ridge regression, and elastic net regression.

## Part One -- Linear Regression as a Baseline

### Questions

1. Begin by fitting a regular linear regression model predicting `BodyFat` using all of the continuous body measurements in the dataset except `Density`. Specifically, the features should be: `Age`, `Weight`, `Height`, `Neck`, `Chest`, `Abdomen`, `Hip`, `Thigh`, `Knee`, `Ankle`, `Biceps`, `Forearm`, and `Wrist`. Call this model `model_linear`.

In [4]:
model_linear = (
    dat
    .pipe(
        DSSklearn.linear_regression,
        target = c.BodyFat,
        features = [c.Age, c.Weight, c.Height, c.Neck, c.Chest, c.Abdomen, c.Hip, c.Thigh, c.Knee, c.Ankle, c.Biceps, c.Forearm, c.Wrist]
    )
)

2. Now print **all 14 rows** (remember `.pipe(print_rows)`) the coefficients from the model.

In [8]:
(
    model_linear.coef()
    .pipe(print_rows)
)

name,param
str,f64
"""Intercept""",19.375568
"""Abdomen""",11.08874
"""Thigh""",1.219733
"""Forearm""",0.902414
"""Biceps""",0.591959
"""Age""",0.40703
"""Ankle""",0.123238
"""Knee""",-0.145316
"""Chest""",-0.161788


3. Now print the RMSE values from the model.

In [9]:
model_linear.score()

{'train': np.float64(4.313259317719465),
 'test': np.float64(3.9599132468772114)}

4. Many of the predictors in this dataset measure related characteristics of a person's body and therefore may contain overlapping information. What variables do you think would be highly related/correlated with each other?

**Answer (Q4):** Weight and Height, Biceps, Forearm and Wrist.

5. Now, explain why having these related or highly correlated predictors could make the coefficients from a regular linear regression model unstable and why a model that fits the training data extremely well may not necessarily provide the best predictions for new observations.

**Answer (Q5):** Because these predictors are related, the cause a problem called multicollinearity. If a model fits the training data extremely well, it's overfitted, which means that it will perform poorly on future data.

## Part Two -- Lasso Regression

### Questions

1. Fit a lasso regression model using the same target and predictor variables as the linear regression model by using `DSSklearn.elastic_net_cv` with `l1_ratio = 1`. Call the resulting model `model_lasso`.

In [10]:
model_lasso = (
    dat
    .pipe(
        DSSklearn.elastic_net_cv,
        target = c.BodyFat,
        features = [c.Age, c.Weight, c.Height, c.Neck, c.Chest, c.Abdomen, c.Hip, c.Thigh, c.Knee, c.Ankle, c.Biceps, c.Forearm, c.Wrist],
        l1_ratio = 1
    )
)

2. Print the training and testing RMSE values for the lasso regression model.

In [11]:
model_lasso.score()

{'train': np.float64(4.315910360733716), 'test': np.float64(3.948433040908772)}

3. using the `alpha()` method, print the penalty value that the model found to be the best for our data.

In [12]:
model_lasso.alpha()

np.float64(0.016819093418042046)

4. Display the standardized coefficients from the lasso model using `coef(raw = True)`. Make sure to again print all 14 rows.

In [14]:
(
    model_linear.coef(raw=True)
    .pipe(print_rows)
)


name,param
str,f64
"""Intercept""",-6.024844
"""Abdomen""",1.008386
"""Forearm""",0.460909
"""Thigh""",0.225919
"""Biceps""",0.1983
"""Ankle""",0.068159
"""Age""",0.032301
"""Chest""",-0.019213
"""Knee""",-0.058833


4. Which predictor variables have coefficients that have been reduced to approximately zero, and which variables retain the largest coefficients?

**Answer (Q4):** Reduced coefficients: Ankle, Age, Chest, Knee, Weight, Height

Largest coefficients: Abdomen, Wrist, Forearm

5. Now, compare the lasso model to the linear regression model done in part 1. Did applying the lasso penalty make the training RMSE better or worse, and did it make the testing RMSE better or worse?

**Answer (Q5):** Slightly worse on training, slightly better on testing

## Part Three -- Ridge Regression

### Questions

1. Now fit a ridge regression model using the same target and predictors by using `DSSklearn.elastic_net_cv` with `l1_ratio = 0`. Also set `alphas = [0, 0.01, 0.1, 0.5, 1, 5, 10]`. Call the model `model_ridge`.

In [15]:
model_ridge = (
    dat
    .pipe(
        DSSklearn.elastic_net_cv,
        target = c.BodyFat,
        features = [c.Age, c.Weight, c.Height, c.Neck, c.Chest, c.Abdomen, c.Hip, c.Thigh, c.Knee, c.Ankle, c.Biceps, c.Forearm, c.Wrist],
        l1_ratio = 0,
        alphas = [0, 0.01, 0.1, 0.5, 1, 5, 10]
    )
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 4221.027037626472, tolerance: 1.0518
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 3751.8627930060456, tolerance: 1.0518
/usr/local/lib/python3.13/dist-packag

2. Print out the training and testing RMSE values for the Ridge regression model.

In [16]:
model_ridge.score()

{'train': np.float64(4.313259317719465),
 'test': np.float64(3.9599132468772167)}

3. Identify the value of alpha selected through cross validation

In [17]:
model_ridge.alpha()

np.float64(0.0)

4. Display the standardized coefficients from the ridge model. Again display all 14 rows.

In [18]:
(
    model_ridge.coef(raw=True)
    .pipe(print_rows)
)

name,param
str,f64
"""Intercept""",-6.024844
"""Abdomen""",1.008386
"""Forearm""",0.460909
"""Thigh""",0.225919
"""Biceps""",0.1983
"""Ankle""",0.068159
"""Age""",0.032301
"""Chest""",-0.019213
"""Knee""",-0.058833


5. Now, do any of the ridge coefficients become nearly zero? Which ones?

**Answer (Q5):** Age, Ankle

## Part Four -- Elastic-Net Regression

### Questions

1. Finally, fit an elastic net regression model using `DSSklearn.elastic_net_cv` with `l1_ratio = 0.5` to give us a good mixture of lasso and ridge regression. Use the same target and feature variabels we have been using. Call the model `model_enet`.

In [20]:
model_enet = (
    dat
    .pipe(
        DSSklearn.elastic_net_cv,
        target = c.BodyFat,
        features = [c.Age, c.Weight, c.Height, c.Neck, c.Chest, c.Abdomen, c.Hip, c.Thigh, c.Knee, c.Ankle, c.Biceps, c.Forearm, c.Wrist],
        l1_ratio = 0.5,
    )
)

2. Print out the training and testing RMSE values for the Elastic-Net regression model.

In [21]:
model_enet.score()

{'train': np.float64(4.323817718420552),
 'test': np.float64(3.9199663207115716)}

3. Identify the value of alpha selected through cross validation

In [22]:
model_enet.alpha()

np.float64(0.01357979408051669)

4. Display the standardized coefficients from the ridge model. Again display all 14 rows.

In [23]:
(
    model_enet.coef(raw=True)
    .pipe(print_rows)
)

name,param
str,f64
"""Intercept""",-4.505427
"""Abdomen""",0.919645
"""Forearm""",0.44469
"""Thigh""",0.206017
"""Biceps""",0.169223
"""Age""",0.046498
"""Ankle""",0.021275
"""Chest""",0.008144
"""Knee""",-0.058432


In [24]:
(
    model_lasso.coef(raw=True)
    .pipe(print_rows)
)

name,param
str,f64
"""Intercept""",-6.189936
"""Abdomen""",0.977726
"""Forearm""",0.436422
"""Thigh""",0.189191
"""Biceps""",0.180159
"""Age""",0.031209
"""Ankle""",0.028602
"""Chest""",-0.0
"""Knee""",-0.02646


In [25]:
(
    model_ridge.coef(raw=True)
    .pipe(print_rows)
)

name,param
str,f64
"""Intercept""",-6.024844
"""Abdomen""",1.008386
"""Forearm""",0.460909
"""Thigh""",0.225919
"""Biceps""",0.1983
"""Ankle""",0.068159
"""Age""",0.032301
"""Chest""",-0.019213
"""Knee""",-0.058833


5. Examine the standardized coefficients from the elastic net model and compare their general pattern with the lasso and ridge coefficients. Does elastic net appear to behave more like lasso, more like ridge, or as a compromise between the two? Use the sizes of the coefficients and whether any have been reduced to zero to support your answer.

**Answer (Q5):** It behaves more like lasso.